# FDS Pokémon Challenge — 68.48% local CV ensemble (Submission 1)

**Purpose:** reproduce the *same* experiment that achieved **68.48% mean accuracy over three stratified folds** on the user's 10,000-battle training data. This is a validation estimate, **not** a promised Kaggle public/private score.

- CatBoost + LightGBM, 65% / 35% probability blend, threshold 0.5.
- Battle features extracted from the competition data with the original `pokemon_features.py` module.
- No external Pokémon databases or pretrained models.
- The module is **cloned from GitHub**, as required by the course.

**Before running:** push the supplied GitHub repository files, replace the URL in the next cell, attach the competition data, including `type_chart_gen2.csv`, and enable Kaggle Internet for `git clone`. Review and adapt the implementation and document AI assistance under your course rules.


## 1. Clone your GitHub code and install dependencies

In [ ]:
# EDIT ONLY THIS LINE WITH YOUR PUBLIC GITHUB REPOSITORY URL
GITHUB_REPO_URL = "https://github.com/YOUR-USERNAME/FDS-Pokemon-Challenge.git"

import sys, subprocess, importlib.util
from pathlib import Path

if "YOUR-USERNAME" in GITHUB_REPO_URL or not GITHUB_REPO_URL.startswith("https://github.com/"):
    raise ValueError("Replace GITHUB_REPO_URL with your actual GitHub repository URL before running.")

# Use a fresh clone on each Kaggle notebook run; never import an old local copy.
REPO_DIR = Path("/kaggle/working/fds_pokemon_code")
if REPO_DIR.exists():
    import shutil
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", "--depth", "1", GITHUB_REPO_URL, str(REPO_DIR)], check=True)
assert (REPO_DIR / "pokemon_features.py").is_file(), "pokemon_features.py missing in the GitHub repo root"

missing = [p for p in ("catboost", "lightgbm") if importlib.util.find_spec(p) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)

sys.path.insert(0, str(REPO_DIR))
print("Imported feature module from:", REPO_DIR)


## 2. Imports and experiment settings — unchanged from the local run

In [ ]:
import sys
import subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, log_loss, confusion_matrix, classification_report

try:
    from catboost import CatBoostClassifier
    from lightgbm import LGBMClassifier, early_stopping, log_evaluation
except ImportError as exc:
    raise ImportError("Install the required packages with `%pip install catboost lightgbm` and restart the kernel.") from exc

RANDOM_STATE = 42
N_SPLITS = 3
from pokemon_features import (read_jsonl, build_species_lookup, build_vocab,
                              load_type_chart, make_features)
print('CatBoost, LightGBM, feature module: loaded')


## 3. Load the competition data

Attach `train.jsonl`, `test.jsonl`, `sample_submission.csv`, and `type_chart_gen2.csv` using **Add Input**. The search below handles different Kaggle dataset folder names.

In [ ]:
DATA_ROOT = Path('/kaggle/input')

# Require all of the competition's main files in the same folder.
folders = sorted({p.parent for p in DATA_ROOT.rglob('train.jsonl')})
valid_folders = [p for p in folders if all((p / name).is_file() for name in ('train.jsonl', 'test.jsonl', 'sample_submission.csv'))]
if len(valid_folders) != 1:
    raise FileNotFoundError(f'Expected exactly one complete competition dataset; found {valid_folders}. Attach your competition data.')
DATA_DIR = valid_folders[0]
train_data = read_jsonl(DATA_DIR / 'train.jsonl')
test_data = read_jsonl(DATA_DIR / 'test.jsonl')
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')
assert len(train_data) == 10000 and len(test_data) == 5000, 'Unexpected dataset size; verify competition files'
assert all('player_won' in b for b in train_data)
assert all('player_won' not in b for b in test_data)
assert set(sample_submission.columns) == {'battle_id', 'player_won'}
print('Data:', DATA_DIR)
print('Battles:', len(train_data), 'train /', len(test_data), 'test')


## 4. Build metadata and extract 714 features

Only training battles contribute to the species lookup and feature vocabulary. The type chart must be the **competition-supplied Gen 2 chart** to reproduce the previous run.

In [ ]:
species_lookup = build_species_lookup(train_data)
vocab = build_vocab(train_data, max_species=55, max_moves=35)

# The chart may be in the competition dataset or an additional attached dataset.
chart_paths = sorted(DATA_ROOT.rglob('type_chart_gen2.csv'))
if len(chart_paths) != 1:
    raise FileNotFoundError(f'Expected one supplied type_chart_gen2.csv, found {len(chart_paths)}. Attach it via Add Input.')
type_chart = load_type_chart(chart_paths[0])
if not type_chart:
    raise ValueError('Type chart could not be loaded; cannot reproduce the previous feature set.')
print('Type chart:', chart_paths[0])
print('Species metadata:', len(species_lookup), '| species indicators:', len(vocab['species']), '| move indicators:', len(vocab['moves']))


In [ ]:
train_features = make_features(train_data, species_lookup, vocab, type_chart)
test_features = make_features(test_data, species_lookup, vocab, type_chart)

# Force precisely the same design matrix, in the same order.
train_features, test_features = train_features.align(test_features, join='left', axis=1, fill_value=0)
assert 'player_won' not in train_features.columns
assert 'battle_id' not in train_features.columns
assert train_features.columns.equals(test_features.columns)

X = train_features.copy()
X_test = test_features.copy()
y = pd.Series([int(b['player_won']) for b in train_data], name='player_won')
train_ids = pd.Series([b['battle_id'] for b in train_data])
test_ids = pd.Series([b['battle_id'] for b in test_data])

if train_ids.duplicated().any() or test_ids.duplicated().any():
    raise ValueError('Duplicate battle_id found; investigate before training')

cat_cols = X.select_dtypes(include=['object', 'string']).columns.tolist()
num_cols = X.columns.difference(cat_cols).tolist()
for col in cat_cols:
    X[col] = X[col].fillna('unknown').astype(str)
    X_test[col] = X_test[col].fillna('unknown').astype(str)
for col in num_cols:
    X[col] = X[col].astype('float32')
    X_test[col] = X_test[col].astype('float32')

print(f'Feature matrix: train {X.shape}, test {X_test.shape}')
print(f'{len(num_cols)} numeric features, {len(cat_cols)} categorical features')
print('Class distribution:', y.value_counts(normalize=True).round(3).to_dict())
display(X[['trend_team_hp_gap_final','trend_active_hp_gap_final',
           'p1_known_fainted','p2_known_fainted','battle_p2_revealed_count']].head())
assert X.shape == (10000, 714), f'Expected the previous feature shape, got {X.shape}'
assert len(cat_cols) == 13, 'Categorical column count differs from the local run'

## 5. Reproduce stratified 3-fold cross-validation

The training parameters, random seed, early stopping, and ensemble weights below are **identical** to the run reporting 68.48% mean fold accuracy. The actual value may differ if data, environment, or library versions differ.

In [ ]:
X_lgb = X.copy()
X_test_lgb = X_test.copy()
for col in cat_cols:
    # Category levels learned from train only; unknown test categories become NaN.
    categories = pd.Index(X[col].unique())
    X_lgb[col] = pd.Categorical(X[col], categories=categories)
    X_test_lgb[col] = pd.Categorical(X_test[col], categories=categories)

CAT_PARAMS = dict(
    iterations=1700, depth=5, learning_rate=0.035, l2_leaf_reg=7.0,
    random_strength=0.7, loss_function='Logloss', eval_metric='Logloss',
    random_seed=RANDOM_STATE, thread_count=4,
    allow_writing_files=False, verbose=False
)
LGB_PARAMS = dict(
    n_estimators=1700, learning_rate=0.025, num_leaves=17,
    min_child_samples=55, colsample_bytree=0.85,
    subsample=0.85, subsample_freq=1, reg_lambda=5.0,
    objective='binary', random_state=RANDOM_STATE, n_jobs=4, verbosity=-1
)

cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
oof_cat = np.zeros(len(y), dtype=float)
oof_lgb = np.zeros(len(y), dtype=float)
cv_rows = []
cat_best_rounds = []
lgb_best_rounds = []

for fold, (tr, va) in enumerate(cv.split(X, y), start=1):
    print(f'\nFold {fold}/{N_SPLITS}')
    cat = CatBoostClassifier(**CAT_PARAMS)
    cat.fit(X.iloc[tr], y.iloc[tr], cat_features=cat_cols,
            eval_set=(X.iloc[va], y.iloc[va]), early_stopping_rounds=150)
    oof_cat[va] = cat.predict_proba(X.iloc[va])[:, 1]
    cat_best_rounds.append(max(50, cat.get_best_iteration() + 1))

    lgb = LGBMClassifier(**LGB_PARAMS)
    lgb.fit(X_lgb.iloc[tr], y.iloc[tr],
            eval_set=[(X_lgb.iloc[va], y.iloc[va])],
            eval_metric='binary_logloss',
            callbacks=[early_stopping(150, verbose=False), log_evaluation(0)])
    oof_lgb[va] = lgb.predict_proba(X_lgb.iloc[va])[:, 1]
    lgb_best_rounds.append(max(50, lgb.best_iteration_))

    p_cat = (oof_cat[va] >= 0.5).astype(int)
    p_lgb = (oof_lgb[va] >= 0.5).astype(int)
    p_blend = (0.65 * oof_cat[va] + 0.35 * oof_lgb[va] >= 0.5).astype(int)
    row = {'fold': fold,
           'catboost_accuracy': accuracy_score(y.iloc[va], p_cat),
           'lightgbm_accuracy': accuracy_score(y.iloc[va], p_lgb),
           'blend_accuracy': accuracy_score(y.iloc[va], p_blend),
           'catboost_rounds': cat_best_rounds[-1],
           'lightgbm_rounds': lgb_best_rounds[-1]}
    cv_rows.append(row)
    print(row)

cv_results = pd.DataFrame(cv_rows)
display(cv_results)
print('\nMean 3-fold accuracy:')
display(cv_results[['catboost_accuracy', 'lightgbm_accuracy', 'blend_accuracy']].agg(['mean', 'std']))

## 6. Verify the ensemble

The expected local mean fold accuracies are about **68.40% CatBoost**, **67.50% LightGBM**, and **68.48% blend**. Do not confuse the local CV score with a Kaggle leaderboard score.

In [ ]:
oof_blend = 0.65 * oof_cat + 0.35 * oof_lgb
print('OOF overall (may differ slightly from mean fold accuracy):')
for name, probs in [('CatBoost', oof_cat), ('LightGBM', oof_lgb), ('Blend', oof_blend)]:
    print(f'{name:12s}: {accuracy_score(y, (probs >= 0.5).astype(int)):.4%}')
print('Mean fold accuracies:')
display(cv_results[['catboost_accuracy', 'lightgbm_accuracy', 'blend_accuracy']].mean().to_frame('mean_accuracy'))


## 7. Final training on all 10,000 battles

As in the original experiment, each final model uses the **median** of its three best fold iteration counts. No test labels are used.

In [ ]:
final_cat_rounds = int(np.median(cat_best_rounds))
final_lgb_rounds = int(np.median(lgb_best_rounds))
print('Final CatBoost rounds:', final_cat_rounds)
print('Final LightGBM rounds:', final_lgb_rounds)

cat_final = CatBoostClassifier(**{**CAT_PARAMS, 'iterations': final_cat_rounds})
cat_final.fit(X, y, cat_features=cat_cols)

lgb_final = LGBMClassifier(**{**LGB_PARAMS, 'n_estimators': final_lgb_rounds})
lgb_final.fit(X_lgb, y)

p_test_cat = cat_final.predict_proba(X_test)[:, 1]
p_test_lgb = lgb_final.predict_proba(X_test_lgb)[:, 1]
p_test_blend = 0.65 * p_test_cat + 0.35 * p_test_lgb
print('Full-data fitting complete.')

## 8. Submission 1 — CatBoost + LightGBM ensemble

This submission uses 714 engineered features from the first 30 turns of each battle. It combines CatBoost and LightGBM predicted probabilities using fixed weights of **0.65** and **0.35**, then applies a **0.5** classification threshold. Model selection used stratified three-fold cross-validation on training battles; the blend had **68.48% mean fold accuracy** in the original local experiment. The models are retrained on all training data before predicting the 5,000 test battles. The public/private leaderboard score may differ.


In [ ]:
# Keep battle IDs aligned with the sample submission (do not sort or assume equal row order).
by_id = pd.Series((p_test_blend >= 0.5).astype(int), index=test_ids.to_numpy())
if not by_id.index.is_unique or not sample_submission['battle_id'].is_unique:
    raise ValueError('Duplicate battle_id detected')
submission = sample_submission.copy()
submission['player_won'] = submission['battle_id'].map(by_id)
if submission['player_won'].isna().any():
    raise ValueError('Some template battle_ids are missing from test predictions')
submission['player_won'] = submission['player_won'].astype(int)
assert submission.shape == (5000, 2)
assert submission.columns.tolist() == ['battle_id', 'player_won']
assert submission['player_won'].isin([0, 1]).all()
OUTPUT = Path('/kaggle/working/submission_blend_6848.csv')
submission.to_csv(OUTPUT, index=False)
print('Saved:', OUTPUT)
display(submission.head())


## 9. Submit the CSV

Find `submission_blend_6848.csv` in the notebook's **Output** files, and submit that file through the competition's submission interface. Saving this notebook version does not itself guarantee that a competition submission was made.

The number **68.48%** refers to your *original mean 3-fold local validation score*, not a promised score on test data. Keep your own report of experiments, document assistance, and follow the university's academic-integrity requirements.
